In [1]:
import pandas as pd

df = pd.read_csv("ElectronicsData.csv")
df.shape

(643, 7)

In [2]:
df = df.dropna(subset=["Price"])
df.shape

(642, 7)

In [3]:
# remove extra characters from the Price column
# remove $ and commas
df["Price"] = df["Price"].str.replace("$", "", regex=False)
df["Price"] = df["Price"].str.replace(",", "", regex=False)
df["Price"] = df["Price"].str.strip()
df["Price"].unique()[:10]

array(['20.99', '18.99', '14.99', '299.99through-449.99',
       '1149.99through-1499.99', '289.99', '399.99', '429.99', '239.97',
       '349.99'], dtype=object)

In [4]:
#keep the first price when there are two
df["Price"] = df["Price"].str.split("through").str[0]
df["Price"].unique()[:10]

array(['20.99', '18.99', '14.99', '299.99', '1149.99', '289.99', '399.99',
       '429.99', '239.97', '349.99'], dtype=object)

In [5]:
# Price is now a number
df["Price"] = df["Price"].astype(float)
df["Price"].dtype

dtype('float64')

In [6]:
#  blank discount means "No Discount"
df["Discount"] = df["Discount"].fillna("No Discount")
df["Discount"].isna().sum()

np.int64(0)

In [7]:
#  keep the number from "After $200 OFF"
df["discount_amount"] = (
    df["Discount"]
    .str.replace(",", "", regex=False)
    .str.extract(r"After \$(\d+) OFF")[0]
    .astype(float)
    .fillna(0)
)
df[["Discount", "discount_amount"]].head(10)

,Discount,discount_amount
0,No Discount,0.0
1,No Discount,0.0
2,No Discount,0.0
3,No Discount,0.0
4,No Discount,0.0
5,No Discount,0.0
6,Special Event - Ends on 01/21/24,0.0
7,Special Event - Ends on 01/21/24,0.0
8,Price valid through 1/21/24,0.0
9,Price valid through 3/31/24,0.0


In [8]:
#  sort discounts into groups
def discount_type(text):
    if "OFF" in text:
        return "Dollars Off"
    elif "valid through" in text or "Special Event" in text:
        return "Limited Time Price"
    elif "Free Shipping" in text:
        return "Free Shipping"
    else:
        return "No Discount"

df["discount_type"] = df["Discount"].apply(discount_type)
df["discount_type"].value_counts()

discount_type
No Discount           442
Dollars Off           107
Limited Time Price     90
Free Shipping           3
Name: count, dtype: int64

In [11]:
# Keep the number of reviews 
df["review_count"] = (
    df["Rating"]
    .str.extract(r"based on (\d+) reviews")[0]
    .fillna(0)
    .astype(int)
)
df[["Rating", "review_count"]].head(10)

,Rating,review_count
0,NaN,0
1,NaN,0
2,NaN,0
3,NaN,0
4,Rated 4.8 out of 5 stars based on 125 reviews.,125
5,NaN,0
6,NaN,0
7,NaN,0
8,Rated 4.2 out of 5 stars based on 238 reviews.,238
9,Rated 4.4 out of 5 stars based on 99 reviews.,99


In [13]:
# keep the star rating 
df["Rating"] = (
    df["Rating"]
    .str.extract(r"Rated ([\d.]+) out of")[0]
    .astype(float)
)
df[["Rating", "review_count"]].head(10)

,Rating,review_count
0,NaN,0
1,NaN,0
2,NaN,0
3,NaN,0
4,4.8,125
5,NaN,0
6,NaN,0
7,NaN,0
8,4.2,238
9,4.4,99


In [14]:
print(df["Rating"].isna().sum())
print((df["review_count"] == 0).sum())

39
39


In [15]:
df["Title"].duplicated().sum()

np.int64(90)

In [16]:
df[df["Title"].duplicated(keep=False)].sort_values("Title")[["Sub Category", "Price", "Title"]].head(10)

,Sub Category,Price,Title
437,Smart Home & Safety,799.99,ADT 13-piece Smart Home Security System with G...
151,Home Security Systems & Cameras,799.99,ADT 13-piece Smart Home Security System with G...
178,Home Security Systems & Cameras,499.99,ADT 9-piece Smart Home Security System with Go...
459,Smart Home & Safety,499.99,ADT 9-piece Smart Home Security System with Go...
100,Gaming Laptops & Notebook Computers,1499.99,"ASUS 16"" ROG Strix G16 Laptop - 13th Gen Intel..."
286,Laptops & Notebook Computers,1499.99,"ASUS 16"" ROG Strix G16 Laptop - 13th Gen Intel..."
106,Gaming Laptops & Notebook Computers,1299.99,"ASUS 16"" ROG Strix G16 Laptop - Intel Core i7-..."
324,Laptops & Notebook Computers,1299.99,"ASUS 16"" ROG Strix G16 Laptop - Intel Core i7-..."
76,Gaming Desktop Computer & PCs,1449.99,ASUS ROG Strix Gaming Desktop - 13th Gen Intel...
29,Desktop Computers & PCs,1449.99,ASUS ROG Strix Gaming Desktop - 13th Gen Intel...


In [17]:
# assign each real product one ID
df["product_id"] = df.groupby(["Title", "Price"]).ngroup() + 1
df["product_id"].nunique()

554

In [18]:

df["original_price"] = df["Price"] + df["discount_amount"]
df[["Price", "discount_amount", "original_price"]][df["discount_amount"] > 0].head()

,Price,discount_amount,original_price
16,1699.99,400.0,2099.99
18,999.99,500.0,1499.99
19,599.99,200.0,799.99
25,499.99,150.0,649.99
26,1599.99,300.0,1899.99


In [19]:
# rename columns and keep only the ones we need
clean = df.rename(columns={
    "Title": "title",
    "Sub Category": "sub_category",
    "Price": "price",
    "Rating": "rating",
})

clean = clean[[
    "product_id", "title", "sub_category", "price", "original_price",
    "discount_amount", "discount_type", "rating", "review_count"
]]
clean.head()

,product_id,title,sub_category,price,original_price,discount_amount,discount_type,rating,review_count
0,126,"Duracell Coppertop Alkaline AA Batteries, 40-c...",Batteries,20.99,20.99,0.0,No Discount,NaN,0
1,127,"Duracell Coppertop Alkaline AAA Batteries, 40-...",Batteries,20.99,20.99,0.0,No Discount,NaN,0
2,124,"Duracell 9V Alkaline Batteries, 8-count",Batteries,20.99,20.99,0.0,No Discount,NaN,0
3,125,"Duracell C Alkaline Batteries, 14-count",Batteries,18.99,18.99,0.0,No Discount,NaN,0
4,123,"Duracell 2032 Lithium Coin Batteries, 12-count",Batteries,14.99,14.99,0.0,No Discount,4.8,125


In [20]:
clean.to_csv("electronics_clean.csv", index=False)
clean.shape

(642, 9)

In [21]:

clean["title"] = clean["title"].str.encode("ascii", "ignore").str.decode("ascii")

clean.to_csv("electronics_clean.csv", index=False)
clean.shape

(642, 9)